### Import the libraries

In [1]:
from binance.client import Client
import pandas as pd
import numpy as np

### Instantiate the Binance client and download the last 1000 BTCEUR trades

In [38]:
spot_client = Client()
trades = spot_client.get_historical_trades(symbol="BTCUSDT", limit=10000)

df = pd.DataFrame(trades)


### Process the downloaded trades into a pandas DataFrame

In [39]:
df.drop(columns=["id", "isBuyerMaker", "isBestMatch"], inplace=True)
df["time"] = pd.to_datetime(df["time"], unit="ms")
for column in ["price", "qty", "quoteQty"]:
    df[column] = pd.to_numeric(df[column])

In [12]:
df.head()

,price,qty,quoteQty,time
0,76000.57,0.00007,5.320040,2026-04-19 14:01:41.813
1,76000.58,0.01290,980.407482,2026-04-19 14:01:41.813
2,76000.58,0.00007,5.320041,2026-04-19 14:01:41.813
3,76000.58,0.00007,5.320041,2026-04-19 14:01:41.813
4,76000.77,0.00007,5.320054,2026-04-19 14:01:41.813


In [40]:
df

,price,qty,quoteQty,time
0,76016.00,0.00040,30.406400,2026-04-19 14:27:20.821
1,76016.00,0.00040,30.406400,2026-04-19 14:27:20.821
2,76016.00,0.00040,30.406400,2026-04-19 14:27:20.821
3,76015.94,0.00008,6.081275,2026-04-19 14:27:20.822
4,76015.71,0.00007,5.321100,2026-04-19 14:27:20.822
...,...,...,...,...
995,75957.05,0.00007,5.316993,2026-04-19 14:28:01.272
996,75957.05,0.00007,5.316993,2026-04-19 14:28:01.272
997,75957.05,0.00007,5.316993,2026-04-19 14:28:01.272
998,75957.05,0.00007,5.316993,2026-04-19 14:28:01.272


###  Define a function aggregating the raw trades information into bars

In [30]:
def get_bars(df, add_time=False):
    ohlc = df["price"].ohlc()
    vwap = (
        df.apply(lambda x: np.average(x["price"], weights=x["qty"]))
        .to_frame("vwap")
    )
    vol = df["qty"].sum().to_frame("vol")
    cnt = df["qty"].size().to_frame("cnt")
    if add_time:
        time = df["time"].last().to_frame("time")
        res = pd.concat([time, ohlc, vwap, vol, cnt], axis=1)
    else:
        res = pd.concat([ohlc, vwap, vol, cnt], axis=1)
    return res

### Get the time bars

In [41]:
df_grouped_time = df.groupby(pd.Grouper(key="time", freq="1Min"))
time_bars = get_bars(df_grouped_time)
time_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_20224\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,open,high,low,close,vwap,vol,cnt
time,,,,,,,
2026-04-19 14:27:00,76016.00,76016.00,75957.04,75957.04,75980.609376,2.95762,980
2026-04-19 14:28:00,75957.04,75957.05,75957.04,75957.05,75957.043850,0.07076,20


### Get the tick bars

In [42]:
bar_size = 100
df["tick_group"] = (
    pd.Series(list(range(len(df))))
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("tick_group")
tick_bars = get_bars(df_grouped_ticks, add_time=True)
tick_bars


C:\Users\MSI VN\AppData\Local\Temp\ipykernel_20224\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
tick_group,,,,,,,,
0,2026-04-19 14:27:20.885,76016.00,76016.00,76010.80,76010.80,76011.950381,0.06196,100
1,2026-04-19 14:27:25.015,76010.80,76010.80,76003.34,76003.34,76003.892586,0.13716,100
2,2026-04-19 14:27:31.289,76003.34,76003.34,75995.55,75995.55,75998.654996,0.33386,100
3,2026-04-19 14:27:31.617,75995.54,75995.54,75986.00,75986.00,75992.890540,0.20120,100
4,2026-04-19 14:27:33.741,75985.03,75985.03,75977.74,75977.75,75984.366294,0.72288,100
5,2026-04-19 14:27:40.165,75977.75,75977.75,75974.00,75974.00,75977.477424,0.37779,100
6,2026-04-19 14:27:56.968,75973.83,75973.83,75970.76,75970.76,75972.390691,0.35300,100
7,2026-04-19 14:27:56.978,75970.76,75970.76,75966.22,75966.22,75970.071397,0.37122,100
8,2026-04-19 14:27:57.013,75966.22,75966.22,75960.10,75960.10,75962.348772,0.04316,100


### Get the volume bars

In [43]:
bar_size = 1
df["cum_qty"] = df["qty"].cumsum()
df["vol_group"] = (
    df["cum_qty"]
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("vol_group")
volume_bars = get_bars(df_grouped_ticks, add_time=True)
volume_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_20224\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
vol_group,,,,,,,,
0,2026-04-19 14:27:31.618,76016.00,76016.00,75984.69,75984.69,75995.547025,0.97957,412
1,2026-04-19 14:27:45.271,75984.69,75984.70,75973.36,75973.37,75980.110916,0.99745,210
2,2026-04-19 14:28:00.836,75973.37,75973.37,75957.04,75957.04,75965.851592,1.01878,366
3,2026-04-19 14:28:01.272,75957.04,75957.05,75957.04,75957.05,75957.041406,0.03258,12


### Get the dollar bars

In [44]:
bar_size = 50000
df["cum_value"] = df["quoteQty"].cumsum()
df["value_group"] = (
    df["cum_value"]
    .div(bar_size)
    .apply(np.floor)
    .astype(int)
    .values
)
df_grouped_ticks = df.groupby("value_group")
dollar_bars = get_bars(df_grouped_ticks, add_time=True)
dollar_bars

C:\Users\MSI VN\AppData\Local\Temp\ipykernel_20224\2030430829.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.apply(lambda x: np.average(x["price"], weights=x["qty"]))


,time,open,high,low,close,vwap,vol,cnt
value_group,,,,,,,,
0,2026-04-19 14:27:31.518,76016.00,76016.00,75992.17,75992.17,76000.291341,0.64162,353
1,2026-04-19 14:27:32.467,75992.17,75992.17,75984.69,75984.69,75985.719581,0.60735,87
2,2026-04-19 14:27:44.892,75984.69,75984.69,75973.36,75973.36,75978.460035,0.72178,180
3,2026-04-19 14:27:57.054,75973.37,75973.37,75960.10,75960.10,75970.126645,0.63162,282
4,2026-04-19 14:28:01.272,75960.10,75960.10,75957.04,75957.05,75958.950099,0.42601,98
